# 05 · Parametric per-N S2 waveform synthesis & classification

            Single notebook that covers what the legacy `2e.ipynb` … `7e.ipynb` series
            +  `*e_cut_wf_gen.py` scripts each duplicated. The electron multiplicity
            is now a parameter (`N_ELECTRONS`).

            Pipeline: CEvNS sim → pattern×ST cut → waveform synthesis →
            (optional) CNN classifier.

In [1]:
# Auto-discover the package even if the notebook is launched from outside the repo.
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 110

In [2]:
N_ELECTRONS = 5  # change this and re-run the rest of the notebook
CONFIG_PATH = '../configs/cevns_sim.yaml'

## 1 – Run a CEvNS batch

In [3]:
from pathlib import Path
from relics_de_sim import DESimConfig, Pipeline
cfg = DESimConfig.from_yaml(CONFIG_PATH)
pipe = Pipeline(cfg, rng=np.random.default_rng(N_ELECTRONS))
muon_files = [Path(cfg.paths.muon_track_dir) / f'muon_track.{i}.npy' for i in range(2)]
pipe.load_muon_tracks(muon_files)
pipe.simulate_cevns()
arr_idx = next(i for i, a in enumerate(pipe.cevns_points)
                if len(a) and int(a['num_e'][0]) == N_ELECTRONS)
arr = pipe.cevns_points[arr_idx]
pe_info = pipe.cevns_pe_info[arr_idx]
print(f'{len(arr)} events with n_e={N_ELECTRONS}')

Reading muon files: 100%|██████████| 2/2 [00:00<00:00, 137.75it/s]
/home/leiyang/TPC_DE_SIm-main/relics_de_sim/recon.py:129: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  st

Using device: cuda
model loaded from epoch /home/leiyang/TPC_DE_SIm-main/model/Regression/DeepResNet25.ckpt ,using device: cuda 


100%|██████████| 237682/237682 [00:00<00:00, 524954.18it/s]


total samples:  237682 , using a 128 batch for data loader


100%|██████████| 1857/1857 [00:04<00:00, 400.30it/s]


...

model loaded from epoch /home/leiyang/TPC_DE_SIm-main/model/Regression/DeepResNet25.ckpt ,using device: cuda 


100%|██████████| 425/425 [00:00<00:00, 369906.45it/s]


total samples:  425 , using a 128 batch for data loader


space-time correlation: 100%|██████████| 1/1 [00:00<00:00, 24.48it/s]

59467 events with n_e=5


## 2 – Apply the pattern × ST cut

In [4]:
from relics_de_sim.cuts import PatternSTCut, calculate_poisson_log_likelihood
cut = PatternSTCut.from_npz(cfg.cuts.k_st_coefficients_path,
                            cfg.cuts.b_coefficients_path)

st_mask = arr['st_cor'] > 0
valid = arr[st_mask]
area = valid['pe_by_area'].sum(axis=1)
log_st_cor = np.log(valid['st_cor'])
pattern_coef = np.sum(
    calculate_poisson_log_likelihood(
        valid['pe_by_area'][:, :64],
        valid['recons_light_pattern'][:, :64]),
    axis=1)
passes = cut.passes(pattern_coef, log_st_cor, area)
print(f'pattern-cut acceptance for n={N_ELECTRONS}: {passes.mean():.3f}')

pattern-cut acceptance for n=5: 0.952


## 3 – Synthesise S2 waveforms for the survivors

In [ ]:
from relics_de_sim.waveform import (
    WaveformParams,
    synthesize_event_waveforms,
)
wf_params = WaveformParams.from_configs(cfg.detector, cfg.electronics)
passing_event_ids = np.where(st_mask)[0][passes]
z_array = np.linspace(0.0, 24.0, int(passes.sum()))
waveforms, stds = synthesize_event_waveforms(
    pe_info, passing_event_ids, z_array, wf_params,
    rng=np.random.default_rng(0))
print('waveforms shape:', waveforms.shape)

fig, ax = plt.subplots(figsize=(8, 3))
for w in waveforms[:5]:
    ax.plot(w[1500:2000])
ax.set_xlabel('sample (centre-trimmed)'); ax.set_ylabel('amplitude (pe/sample)')
ax.set_title(f'first 5 surviving waveforms, n_e={N_ELECTRONS}')
plt.show()

## 4 – (Optional) classifier

            Skip if you don't have ``models/waveform_classifier.pth`` or PyTorch.

In [ ]:
try:
    from relics_de_sim.waveform import WaveformClassifier
    clf = WaveformClassifier(cfg.paths.waveform_classifier_model)
    scores = clf(waveforms)
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.hist(scores, bins=40)
    ax.set_xlabel('classifier score'); ax.set_ylabel('count')
    plt.show()
except (ImportError, FileNotFoundError, Exception) as exc:  # pragma: no cover
    print('Skipping classifier:', exc)

Skipping classifier: Error(s) in loading state_dict for _Conv1dNet:
	Missing key(s) in state_dict: "net.0.weight", "net.0.bias", "net.3.weight", "net.3.bias", "net.6.weight", "net.6.bias", "net.10.weight", "net.10.bias". 
	Unexpected key(s) in state_dict: "conv1.0.weight", "conv1.0.bias", "conv1.1.weight", "conv1.1.bias", "conv1.1.running_mean", "conv1.1.running_var", "conv1.1.num_batches_tracked", "conv2.0.weight", "conv2.0.bias", "conv2.1.weight", "conv2.1.bias", "conv2.1.running_mean", "conv2.1.running_var", "conv2.1.num_batches_tracked", "conv3.0.weight", "conv3.0.bias", "conv3.1.weight", "conv3.1.bias", "conv3.1.running_mean", "conv3.1.running_var", "conv3.1.num_batches_tracked", "fc.0.weight", "fc.0.bias", "fc.1.weight", "fc.1.bias", "fc.1.running_mean", "fc.1.running_var", "fc.1.num_batches_tracked", "fc.4.weight", "fc.4.bias", "fc.5.weight", "fc.5.bias", "fc.5.running_mean", "fc.5.running_var", "fc.5.num_batches_tracked", "fc.8.weight", "fc.8.bias". 


/home/leiyang/TPC_DE_SIm-main/relics_de_sim/waveform.py:198: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = self._torch.load(checkpoint_path, map_location=self.device